In [ ]:
# === random_forest.ipynb — Setup ===
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Load the FROZEN dataset — same file the logistic regression used
# (lives in the central data/model_dataset/ folder, shared by every model)
model_df = pd.read_csv('../../data/model_dataset/model_dataset.csv')

pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
y_all = model_df['burned'].astype(int)

print("Dataset:", model_df.shape, "| burned:", int(y_all.sum()),
      f"({100*y_all.mean():.1f}%)")
print("Predictors:", pred_cols)

Dataset: (6231, 13) | burned: 2077 (33.3%)
Predictors: ['dist_roads', 'dist_parks', 'dist_coca', 'dist_mosaic', 'temp_C', 'vpd_kPa', 'ndvi', 'wind_ms', 'oni']


In [ ]:
# ---------- 0. Evaluation function ----------
def evaluate(y_true, prob, pred):
    return (roc_auc_score(y_true, prob),
            average_precision_score(y_true, prob),
            f1_score(y_true, pred))

def make_rf():
    return RandomForestClassifier(
        n_estimators=300, max_features='sqrt', min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1)

X = model_df[pred_cols].values

# ---------- 1. SPATIAL BLOCK CV ----------
BLOCK = 0.25
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                    (model_df['lat']//BLOCK).astype(int).astype(str)
groups = model_df['block'].values
gkf = GroupKFold(n_splits=5)

rows = []
for fold,(tr,te) in enumerate(gkf.split(X, y_all, groups), 1):
    m = make_rf().fit(X[tr], y_all[tr])            # trees are scale-invariant — no scaler
    prob = m.predict_proba(X[te])[:,1]; pred = m.predict(X[te])
    auc, prauc, f1 = evaluate(y_all[te], prob, pred)
    rows.append((auc, prauc, f1))
    print(f"  Fold {fold}: AUC={auc:.3f}  PR-AUC={prauc:.3f}  F1={f1:.3f}")

rows = np.array(rows)
print(f"\nSPATIAL BLOCK CV — Random Forest")
print(f"  AUC-ROC : {rows[:,0].mean():.3f} ± {rows[:,0].std():.3f}")
print(f"  PR-AUC  : {rows[:,1].mean():.3f} ± {rows[:,1].std():.3f}")
print(f"  F1      : {rows[:,2].mean():.3f} ± {rows[:,2].std():.3f}")

# ---------- 2. TEMPORAL SPLIT ----------
tr = (model_df['year'] <= 2019).values
te = (model_df['year'] >= 2020).values
m = make_rf().fit(X[tr], y_all[tr])
prob = m.predict_proba(X[te])[:,1]; pred = m.predict(X[te])
auc, prauc, f1 = evaluate(y_all[te], prob, pred)

print(f"\nTEMPORAL SPLIT — Random Forest")
print(f"  train ≤2019: {tr.sum()} rows ({int(y_all[tr].sum())} events) | "
      f"test ≥2020: {te.sum()} rows ({int(y_all[te].sum())} events)")
print(f"  AUC-ROC : {auc:.3f}")
print(f"  PR-AUC  : {prauc:.3f}")
print(f"  F1      : {f1:.3f}")

# ---------- 3. Feature importance (impurity-based — SHAP comes later) ----------
m_all = make_rf().fit(X, y_all)
imp = pd.DataFrame({'predictor': pred_cols, 'importance': m_all.feature_importances_}) \
        .sort_values('importance', ascending=False)
print("\nRF impurity importance")
print(imp.round(3).to_string(index=False))

  Fold 1: AUC=0.907  PR-AUC=0.841  F1=0.768
  Fold 2: AUC=0.840  PR-AUC=0.653  F1=0.598
  Fold 3: AUC=0.881  PR-AUC=0.814  F1=0.747
  Fold 4: AUC=0.882  PR-AUC=0.796  F1=0.734
  Fold 5: AUC=0.867  PR-AUC=0.741  F1=0.700

SPATIAL BLOCK CV — Random Forest
  AUC-ROC : 0.875 ± 0.022
  PR-AUC  : 0.769 ± 0.067
  F1      : 0.709 ± 0.060

TEMPORAL SPLIT — Random Forest
  train ≤2019: 5153 rows (1828 events) | test ≥2020: 1078 rows (249 events)
  AUC-ROC : 0.817
  PR-AUC  : 0.563
  F1      : 0.549

RF impurity importance
  predictor  importance
       ndvi       0.222
    wind_ms       0.201
    vpd_kPa       0.162
     temp_C       0.109
 dist_parks       0.082
        oni       0.075
 dist_roads       0.068
  dist_coca       0.046
dist_mosaic       0.035
